In [7]:
#!pip install -U langchain langchain-openai faiss-cpu
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI

# langchain-openai --> framework-package

In [8]:
# 1. Load documents
documents = [
    "RAG combines retrieval with generation.",
    "Vector databases store embeddings.",
    "Agents can use RAG for knowledge."
]


In [10]:
# 2. Create embeddings
embeddings = OpenAIEmbeddings(openai_api_key="YOUR_API_KEY")
# YOUR_API_KEY - This is your authentication key to call OpenAI services. without this your code fails with openai.AuthenticationError: No API key provided.
# correct way is to set env varaible and call export OPENAI_API_KEY="your_real_key".
# When OpenAIEmbeddign is called it interally makes an HTTPS call to POST https://api.openai.com/v1/embeddings
#Authorization: Bearer YOUR_API_KEY

In [11]:
# Sample data
texts = ["LangChain is powerful", "RAG pipelines are useful"]

# Create vector store
vectorstore = FAISS.from_texts(texts, embeddings)
# FAISS vector store converts text into embeddings and stores them for fast similarity-based retrieval.
# Open API call is also happening here

#FAISS.from_texts(...)
#        ↓
#calls → embeddings.embed_documents(texts)
#        ↓
#calls → OpenAI API (for each text)
#        ↓
#returns vectors
#        ↓
#stored in FAISS index


# this is what happens behind the text
# Step 1: Convert text → embeddings (API call happens HERE)
#vectors = embeddings.embed_documents(texts)

# Step 2: Store vectors in FAISS
#vectorstore = FAISS.from_embeddings(vectors, texts)

# Query
docs = vectorstore.similarity_search("What is LangChain?")

#Query → embedding (API call again)
#      → vector
#      → similarity search (FAISS - local, fast) - FAISS introduced by Meta for approximate nearest neighbor (ANN) search
#      → returns closest documents


print(docs)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_API_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [14]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Load local embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# Sample data
texts = ["LangChain is powerful", "RAG pipelines are useful"]

# Create vector store (NO API CALL)
vectorstore = FAISS.from_texts(texts, embeddings)

# Query
docs = vectorstore.similarity_search("What is LangChain?")
print(docs)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7245.38it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Document(id='4f346ad4-faa8-4bbe-b311-e18b62d34952', metadata={}, page_content='LangChain is powerful'), Document(id='2af2d3f4-3e5d-4387-9adc-3bf62eea14ba', metadata={}, page_content='RAG pipelines are useful')]


In [12]:
# =========================
# 1. Imports (Fixed for v1.2+)
# =========================
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate

# In v1.2+, these are now in langchain.chains or langchain_classic.chains
# If 'langchain.chains' still fails, use 'pip install langchain-classic'
# and change these to 'from langchain_classic.chains import ...'
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# =========================
# 2. Local Embeddings
# =========================
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# =========================
# 3. Sample Data
# =========================
texts = [
    "LangChain is a framework for building LLM applications.",
    "RAG stands for Retrieval Augmented Generation.",
    "FAISS is used for fast vector similarity search."
]

# =========================
# 4. Vector Store (FAISS)
# =========================
vectorstore = FAISS.from_texts(texts, embeddings)
retriever = vectorstore.as_retriever()

# =========================
# 5. Local LLM (Ollama - Updated to OllamaLLM)
# =========================
llm = OllamaLLM(model="llama3") # Ensure you ran: ollama pull llama3

# =========================
# 6. Prompt
# =========================
prompt = ChatPromptTemplate.from_template(
    "Answer ONLY using the context below.\n\n"
    "Context:\n{context}\n\n"
    "Question: {input}"
)

# =========================
# 7. Retrieval Chain (Fixed Syntax)
# =========================
# Step A: Create the chain that handles the document formatting
combine_docs_chain = create_stuff_documents_chain(llm, prompt)

# Step B: Create the final retrieval chain
qa_chain = create_retrieval_chain(retriever, combine_docs_chain)

# =========================
# 8. Run Query
# =========================
response = qa_chain.invoke({"input": "What is RAG?"})

# The key for the answer in the new chains is usually 'answer'
print("\nAnswer:\n", response["answer"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7436.20it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Answer:
 RAG stands for Retrieval Augmented Generation.


In [9]:
!pip show langchain langchain-community langchain-core

Name: langchain
Version: 1.2.15
Summary: Building applications with LLMs through composability
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /Users/manish.singh/anaconda3/lib/python3.11/site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-community
Version: 0.4.1
Summary: Community contributed LangChain integrations.
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /Users/manish.singh/anaconda3/lib/python3.11/site-packages
Requires: aiohttp, dataclasses-json, httpx-sse, langchain-classic, langchain-core, langsmith, numpy, pydantic-settings, PyYAML, requests, SQLAlchemy, tenacity
Required-by: 
---
Name: langchain-core
Version: 1.2.26
Summary: Building applications with LLMs through composability
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /Users/manish.singh/anaconda3/lib/python3.11/site-packages
Requires: jsonpatch, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions, uuid-utils
Required-by: 